# LUNAR – analiza wyników (agregacje czytelne do porównań)

Ten notebook zastępuje jeden zbiorczy boxplot kilkoma **mniejszymi, tematycznymi porównaniami**,
każde w formie: **tabela + odpowiednio dobrany wykres** (nie tylko boxplot – także krzywe:
PR-curve, krzywe wrażliwości na szum/imbalance/feature-dropout).

Struktura (6 sekcji):
1. Wyniki samego LUNAR (PR-curve + metryki wg wariantu progu)
2. LUNAR (bez fusion/ensemble/robustness) vs LOF / IsolationForest / OneClassSVM
3. Ensemble z LUNAR vs bez LUNAR (+ LUNAR Self-Ensemble)
4. Fusion (poziomy fuzji + strategie + krzywa wrażliwości na szum/imbalance)
5. Robustness (krzywe F1/AUC vs feature_dropout, imbalance, noise)
6. Końcowe porównanie – najlepsze warianty z każdej rodziny

**Zanim uruchomisz:** ustaw poniżej `RESULTS_DIR` (folder z plikami `*.json` pojedynczych runów,
tak jak w `aggregate_results.py`) oraz `AGG_DIR` (folder z `all_results_flat.csv` i
`summary_by_dataset_model_fusion_variant.csv`). Każdy wykres jest od razu wyświetlany (`plt.show()`)
i zapisywany równocześnie jako **PDF** i **PNG** do `FIG_DIR`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# ------------------------------------------------------------------
# KONFIGURACJA ŚCIEŻEK — dostosuj do swojego układu katalogów
# ------------------------------------------------------------------
RESULTS_DIR = Path("results")                 # folder z pojedynczymi *.json (surowe runy, z pr_curve)
AGG_DIR     = RESULTS_DIR / "aggregation"      # folder z CSV-ami wygenerowanymi przez aggregate_results.py
FIG_DIR     = AGG_DIR / "notebook_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 110,
})
PALETTE = "Set2"

def save_fig(fig, name):
    """Zapisuje figurę jako PNG (300 dpi) i PDF do FIG_DIR, zwraca ścieżki."""
    png_path = FIG_DIR / f"{name}.png"
    pdf_path = FIG_DIR / f"{name}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Zapisano: {png_path.name} , {pdf_path.name}")
    return png_path, pdf_path

def load_raw_records(results_dir):
    """Wczytuje surowe pliki *.json (z polami pr_curve / threshold_variants),
    tak jak robi to aggregate_results.py."""
    records = []
    for path in sorted(Path(results_dir).glob("*.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                records.append(json.load(f))
        except Exception as e:
            print("SKIP", path.name, e)
    return records

In [ ]:
# ------------------------------------------------------------------
# WCZYTANIE DANYCH
# ------------------------------------------------------------------
df = pd.read_csv(AGG_DIR / "all_results_flat.csv")

# tylko wiersze z domyślnym (F1-max) progiem do większości porównań tabelarycznych
df_std = df[df["threshold_variant"] == "standard"].copy()

METRIC_COLS = ["AUC_ROC", "AUC_PR", "Precision", "Recall", "F1"]

print(df.shape, "wierszy w all_results_flat.csv")
print("Dostępne model_type:", sorted(df["model_type"].unique()))
print("Dostępne dataset_name:", sorted(df["dataset_name"].unique()))

## 1. Wyniki samego LUNAR

Tabela metryk LUNAR (bez fuzji/ensemble/robustness) dla każdego datasetu i wariantu progu,
oraz **krzywa Precision-Recall** (z surowego JSON, jeśli plik jest dostępny w `RESULTS_DIR`)
zamiast boxplota — dla pojedynczego modelu boxplot i tak nic nie pokazuje (1 punkt danych).

In [ ]:
# --- Tabela ---
lunar_tbl = df[df["model_type"] == "LUNAR"][
    ["dataset_name", "threshold_variant", "AUC_ROC", "AUC_PR", "Precision", "Recall", "F1",
     "runtime_train", "runtime_inference"]
].sort_values(["dataset_name", "threshold_variant"]).reset_index(drop=True)

display(lunar_tbl.style.format({c: "{:.3f}" for c in METRIC_COLS} |
                                {"runtime_train": "{:.1f}", "runtime_inference": "{:.2f}"})
        .set_caption("LUNAR (model podstawowy) – metryki wg datasetu i wariantu progu"))

lunar_tbl.to_csv(FIG_DIR / "table_1_lunar_alone.csv", index=False)

In [ ]:
# --- Wykres: krzywe PR dla LUNAR, jedna linia na dataset ---
records = load_raw_records(RESULTS_DIR)
lunar_records = [r for r in records if r.get("model_type") == "LUNAR" and r.get("pr_curve")]

fig, ax = plt.subplots(figsize=(7, 5.5))
colors = sns.color_palette(PALETTE, max(len(lunar_records), 3))

if lunar_records:
    for i, rec in enumerate(lunar_records):
        pr = pd.DataFrame(rec["pr_curve"])
        ax.plot(pr["recall"], pr["precision"], marker="o", markersize=3,
                color=colors[i], label=f'{rec["dataset_name"]} (AUC_PR={rec.get("AUC_PR", float("nan")):.3f})')
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("LUNAR — krzywa Precision-Recall")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.legend(title="Dataset")
else:
    # fallback: brak pr_curve w RESULTS_DIR -> pokaż metryki wg wariantu progu jako wykres słupkowy
    plot_df = lunar_tbl.melt(id_vars=["dataset_name", "threshold_variant"],
                              value_vars=["Precision", "Recall", "F1"],
                              var_name="metric", value_name="value")
    sns.barplot(data=plot_df, x="threshold_variant", y="value", hue="metric",
                palette=PALETTE, ax=ax)
    ax.set_title("LUNAR — Precision/Recall/F1 wg wariantu progu\n"
                  "(brak pr_curve.json w RESULTS_DIR — pokazano metryki progowe zamiast krzywej PR)")
    ax.set_ylabel("Wartość")

fig.tight_layout()
save_fig(fig, "01_lunar_alone_pr_curve")
plt.show()

## 2. LUNAR (bez fusion/ensemble/robustness) vs modele podstawowe (LOF, IsolationForest, OneClassSVM)

Wykres słupkowy grupowany wg datasetu, z modelem na osi X — czytelny odpowiednik boxplota
dla małej liczby pojedynczych wyników (po jednym runie na model/dataset).

In [ ]:
base_models = ["LUNAR", "LOF", "IsolationForest", "OneClassSVM"]
base_df = df_std[df_std["model_type"].isin(base_models)].copy()

base_tbl = base_df[["dataset_name", "model_type", "AUC_ROC", "AUC_PR", "Precision", "Recall", "F1",
                     "runtime_train", "runtime_inference"]].sort_values(["dataset_name", "model_type"])
display(base_tbl.style.format({c: "{:.3f}" for c in METRIC_COLS} |
                               {"runtime_train": "{:.1f}", "runtime_inference": "{:.2f}"})
        .set_caption("LUNAR vs modele podstawowe (próg standardowy / F1-max)"))
base_tbl.to_csv(FIG_DIR / "table_2_lunar_vs_baselines.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
for ax, metric in zip(axes, ["AUC_ROC", "F1"]):
    sns.barplot(data=base_df, x="model_type", y=metric, hue="dataset_name",
                order=base_models, palette=PALETTE, ax=ax)
    ax.set_title(f"{metric} — LUNAR vs modele podstawowe")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=15)
axes[0].legend(title="Dataset")
axes[1].get_legend().remove()
fig.tight_layout()
save_fig(fig, "02_lunar_vs_baselines")
plt.show()

## 3. Ensemble z LUNAR vs bez LUNAR (+ LUNAR Self-Ensemble)

Porównanie: `Ensemble_with_LUNAR` vs `Ensemble_without_LUNAR` (klasyczny ensemble bazowych
detektorów, z LUNAR jako dodatkowym członem lub bez) oraz warianty `LUNAR_Self_Ensemble`
(bagging/boosting/stacking/voting samego LUNAR).

In [ ]:
ens_models = ["Ensemble_with_LUNAR", "Ensemble_without_LUNAR", "LUNAR_Self_Ensemble"]
ens_df = df_std[df_std["model_type"].isin(ens_models)].copy()
ens_df["variant"] = ens_df["model_type"] + ens_df["fusion_strategy"].apply(lambda s: f" ({s})" if pd.notna(s) else "")

ens_tbl = ens_df[["dataset_name", "model_type", "fusion_strategy", "AUC_ROC", "AUC_PR",
                   "Precision", "Recall", "F1"]].sort_values(["dataset_name", "model_type", "fusion_strategy"])
display(ens_tbl.style.format({c: "{:.3f}" for c in METRIC_COLS})
        .set_caption("Ensemble z LUNAR vs bez LUNAR (+ Self-Ensemble) — próg standardowy"))
ens_tbl.to_csv(FIG_DIR / "table_3_ensemble.csv", index=False)

fig, axes = plt.subplots(1, len(df_std["dataset_name"].unique()), figsize=(14, 6), sharey=True)
for ax, (dset, sub) in zip(np.atleast_1d(axes), ens_df.groupby("dataset_name")):
    sub = sub.sort_values("F1", ascending=False)
    sns.barplot(data=sub, y="variant", x="F1", hue="model_type", dodge=False,
                palette=PALETTE, ax=ax)
    ax.set_title(dset)
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1")
    ax.set_ylabel("")
    if ax.get_legend():
        ax.legend(title="Rodzina", loc="lower right", fontsize=8)
fig.suptitle("F1 dla wariantów ensemble, wg datasetu")
fig.tight_layout()
save_fig(fig, "03_ensemble_with_vs_without_lunar")
plt.show()

## 4. Fusion — poziomy i strategie fuzji

`Fusion_Level_Comparison` (decision_level_AND/OR, feature_level, score_level) oraz
`LUNAR_Fusion_v2` (rank_mean / stacking_lr / max) wraz z **krzywą wrażliwości F1 na poziom
szumu i nierównowagi klas** (parametry `noise`, `imbalance_factor_test`) — to zastępuje
nieczytelny boxplot krzywą zależności metryki od parametru.

In [ ]:
fusion_level_df = df_std[df_std["model_type"] == "Fusion_Level_Comparison"].copy()
fusion_tbl = fusion_level_df[["dataset_name", "fusion_strategy", "AUC_ROC", "AUC_PR",
                               "Precision", "Recall", "F1"]].sort_values(["dataset_name", "fusion_strategy"])
display(fusion_tbl.style.format({c: "{:.3f}" for c in METRIC_COLS})
        .set_caption("Fusion_Level_Comparison — porównanie poziomów fuzji (próg standardowy)"))
fusion_tbl.to_csv(FIG_DIR / "table_4a_fusion_levels.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=fusion_level_df, x="fusion_strategy", y="F1", hue="dataset_name",
            palette=PALETTE, ax=ax)
ax.set_title("F1 wg poziomu fuzji")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=20)
ax.set_xlabel("")
fig.tight_layout()
save_fig(fig, "04a_fusion_levels_bar")
plt.show()

In [ ]:
# --- Krzywa wrażliwości LUNAR_Fusion_v2 na szum i imbalance (zamiast boxplota) ---
fv2 = df_std[df_std["model_type"] == "LUNAR_Fusion_v2"].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# odczyt parametrów noise/imbalance z kolumny hyperparameters (string reprezentujący dict)
import ast
def get_param(hp_str, key):
    try:
        d = ast.literal_eval(hp_str)
        return d.get(key, np.nan)
    except Exception:
        return np.nan

fv2["noise"] = fv2["hyperparameters"].apply(lambda s: get_param(s, "noise"))
fv2["imbalance_factor_test"] = fv2["hyperparameters"].apply(lambda s: get_param(s, "imbalance_factor_test"))

for ax, param in zip(axes, ["noise", "imbalance_factor_test"]):
    curve = fv2.dropna(subset=[param]).groupby(["dataset_name", param])["F1"].mean().reset_index()
    for dset, sub in curve.groupby("dataset_name"):
        sub = sub.sort_values(param)
        ax.plot(sub[param], sub["F1"], marker="o", label=dset)
    ax.set_xlabel(param)
    ax.set_ylabel("F1")
    ax.set_title(f"LUNAR_Fusion_v2: F1 vs {param}")
    ax.set_ylim(0, 1)
    ax.legend(title="Dataset")

fig.tight_layout()
save_fig(fig, "04b_fusion_v2_sensitivity_curves")
plt.show()

fv2_tbl = fv2[["dataset_name", "fusion_strategy", "noise", "imbalance_factor_test",
               "AUC_ROC", "F1"]].sort_values(["dataset_name", "fusion_strategy", "noise", "imbalance_factor_test"])
display(fv2_tbl.style.format({"AUC_ROC": "{:.3f}", "F1": "{:.3f}"})
        .set_caption("LUNAR_Fusion_v2 — F1/AUC_ROC dla różnych poziomów szumu / imbalance"))
fv2_tbl.to_csv(FIG_DIR / "table_4b_fusion_v2_sensitivity.csv", index=False)

## 5. Robustness — krzywe wrażliwości

`Robustness_Analysis` bada wpływ `feature_dropout`, `imbalance_factor_test` i `noise` na
jakość modelu. Zamiast boxplota — **krzywe F1/AUC_ROC w funkcji parametru zaburzenia**,
z uśrednieniem po powtórzeniach (błędy std tam, gdzie `count > 1` w pliku summary).

In [ ]:
rob = df_std[df_std["model_type"] == "Robustness_Analysis"].copy()
rob["feature_dropout"] = rob["hyperparameters"].apply(lambda s: get_param(s, "feature_dropout"))
rob["imbalance_factor_test"] = rob["hyperparameters"].apply(lambda s: get_param(s, "imbalance_factor_test"))
rob["noise"] = rob["hyperparameters"].apply(lambda s: get_param(s, "noise"))

params = ["feature_dropout", "imbalance_factor_test", "noise"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, param in zip(axes, params):
    sub_all = rob.dropna(subset=[param])
    if sub_all.empty:
        ax.set_title(f"{param}: brak danych")
        continue
    agg = sub_all.groupby(["dataset_name", "fusion_strategy", param])["F1"].agg(["mean", "std"]).reset_index()
    for (dset, strat), g in agg.groupby(["dataset_name", "fusion_strategy"]):
        g = g.sort_values(param)
        ax.errorbar(g[param], g["mean"], yerr=g["std"], marker="o", capsize=3,
                    label=f"{dset} / {strat}")
    ax.set_xlabel(param)
    ax.set_title(f"F1 vs {param}")
    ax.set_ylim(0, 1)

axes[0].set_ylabel("F1")
axes[-1].legend(title="Dataset / strategia", fontsize=8, loc="lower left")
fig.suptitle("Robustness_Analysis — krzywe wrażliwości F1")
fig.tight_layout()
save_fig(fig, "05_robustness_curves")
plt.show()

rob_tbl = rob[["dataset_name", "fusion_strategy", "feature_dropout", "imbalance_factor_test",
               "noise", "AUC_ROC", "F1"]].sort_values(["dataset_name", "fusion_strategy"])
display(rob_tbl.style.format({"AUC_ROC": "{:.3f}", "F1": "{:.3f}"})
        .set_caption("Robustness_Analysis — pełna tabela"))
rob_tbl.to_csv(FIG_DIR / "table_5_robustness.csv", index=False)

## 6. Końcowe porównanie

Zestawienie najlepszego wariantu z każdej rodziny modeli (wg F1, próg standardowy) —
jeden czytelny wykres słupkowy + tabela podsumowująca cały eksperyment.

In [ ]:
family_map = {
    "LUNAR": "LUNAR (solo)",
    "LOF": "LOF (baseline)",
    "IsolationForest": "IsolationForest (baseline)",
    "OneClassSVM": "OneClassSVM (baseline)",
    "Ensemble_with_LUNAR": "Ensemble +LUNAR",
    "Ensemble_without_LUNAR": "Ensemble -LUNAR",
    "LUNAR_Self_Ensemble": "LUNAR Self-Ensemble",
    "Fusion_Level_Comparison": "Fusion (best level)",
    "LUNAR_Fusion_v2": "Fusion v2 (robust)",
    "LUNAR_plus_BestBaseline": "LUNAR+BestBaseline",
    "Robustness_Analysis": "Robustness (best cfg)",
}

best_rows = []
for dset, sub in df_std.groupby("dataset_name"):
    for model_type, label in family_map.items():
        cand = sub[sub["model_type"] == model_type]
        if cand.empty:
            continue
        best = cand.loc[cand["F1"].idxmax()]
        best_rows.append({
            "dataset_name": dset, "family": label, "model_type": model_type,
            "fusion_strategy": best.get("fusion_strategy", np.nan),
            "AUC_ROC": best["AUC_ROC"], "AUC_PR": best["AUC_PR"],
            "Precision": best["Precision"], "Recall": best["Recall"], "F1": best["F1"],
        })

final_tbl = pd.DataFrame(best_rows).sort_values(["dataset_name", "F1"], ascending=[True, False])
display(final_tbl.style.format({c: "{:.3f}" for c in METRIC_COLS})
        .set_caption("Najlepszy wariant z każdej rodziny modeli (wg F1, próg standardowy)"))
final_tbl.to_csv(FIG_DIR / "table_6_final_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 7))
order = final_tbl.groupby("family")["F1"].mean().sort_values(ascending=False).index
sns.barplot(data=final_tbl, y="family", x="F1", hue="dataset_name", order=order,
            palette=PALETTE, ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel("F1 (najlepszy wariant rodziny)")
ax.set_ylabel("")
ax.set_title("Końcowe porównanie rodzin modeli")
ax.legend(title="Dataset")
fig.tight_layout()
save_fig(fig, "06_final_comparison")
plt.show()